<img src="./logo_UTN.svg" align="right" width="150" />

# Tarea Semanal 1

*Alumno: Lapilli Cañedo, Mariano Adrian*

## Consigna

Dadas las siguientes topologías circuitales de un filtro Pasa Banda Pasivo: 

<img src="./fig.png" align="centre" width="500" /> 

Se propone reemplazar el Inductor del filtro de la Fig. 1 con un GIC:


<img src="./GIC.png" align="centre" width="500" /> 

1. Analizar la impedancia de entrada desde el nodo Vx. Hallar los valores de R4,R5,Y1,Y2 e Y3 de tal manera que responda como un Inductor de valor unitario. 

2. Obtener la función transferencia $ \frac{V1}{V2} $ Desnormalice convenientemente en frecuencia e impedancia para garantizar una 
 $fo = 10KHz$ y $ Q = 20 $ utilizando Capacitores entre 1 nF y 100 nF. Para los resistores no hay restricciones. 

3.  Simule la transferencia desnormalizada en Python. Se sugiere programar la transferencia normalizada y definir las normas de frecuencia e impedancia para desnormalizar. Verificar los requerimientos pedidos en 2)


In [ ]:
# Importo librerias de python para trabajar

import numpy as np
from scipy import signal as sig
import sympy as sp
from matplotlib import pyplot as plt
from pytc2.remociones import remover_polo_dc
from pytc2.general import a_equal_b_latex_s, print_latex, s, symbfunc2tf, factorSOS
from pytc2.sistemas_lineales import bodePlot, pzmap, GroupDelay, analyze_sys


Matplotlib is building the font cache; this may take a moment.


## Item 1:

Para este item lo que vamos a hacer es analizar la impedancia que se ve desde el nodo Vx, por eso vamos a definir con que nodos vamos a operar:

<img src="./GICn.png" align="centre" width="500" /> 

Los nodos marcados con verde son con los que podemos armar un sistema de ecuaciones basandonos en las corrientes salientes de los mismos (siempre teniendo en cuenta que las corrientes no ingresan en los operacionales al ser considerados ideales). Los nodos marcados en violeta no se pueden utilizar porque están relacionados con las salidas de los operaciones y no podemos determinar que corriente sale de estos. Luego, pudiendo asumir que ambos OPAMPs están realimentados negativamente asumimos un corto circuito virtual. Aclarado esto, de esta forma quedan las ecuaciones de nodo:

$Nodo \ A:$
$\frac{(V_1 - V_x)}{R} + V_x \ sC + \frac{(V_x - V_B)}{Y_1} = 0$

$Nodo \ B:$
$\frac{(V_x - V_B)}{Y_2} + \frac{(V_x - V_2)}{Y_3}  = 0$

$Nodo \ C:$
$\frac{(V_x - V_2)}{R_4} + \frac{(V_x - V_2)}{R_5}  = 0$

**Nota:** Si bien el nodo A esta bien planteado, solo tomo la corriente de entrada (La que pasa por Y1).

Del nodo C:

$$ V_2 = V_x \ (1 + \frac{R_4}{R_5}) $$

Del nodo B: 

$$ V_B = V_x (1 + \frac{Y_2}{Y_3}) - V_x (\frac{Y_2}{Y_3} + \frac{Y_2 Y_4}{Y_3 Y_5})$$

Haciendo uso del álgreba llego a que:

$$ Z_{in} = \frac{V_x}{I_{in}} = \frac{Z_1 Z_3 R_5}{Z_2 R_4} $$

Finalmente para que $ Z_{in} = s L_x $

$Z_2 = \frac{1}{sC}$

$Z_1 \ Z_3 \ Z_5 = \frac{sC}{R_4}$

<img src="./item1.png" align="centre" width="1000" /> 


## Item 2 

Para este item lo que hice fue utilizar el circuito antes de utilizar el girador y lo reemplacé luego por su impedancia equivalente.

Operando llegué a que:

$$ T(s) = \frac{s \ \frac{1}{C_1 R} }{s^2 + 1 \frac{1}{C_1 R} + \frac{1}{C_1 L} } $$

Teniendo en cuenta que $L = \frac{Z_1 Z_3 R_5}{Z_2 R_4}$ y forzando que el capacitor $ C_1 = 47 nF$ puedo despejar el valor de L para que la $f_o$ valga $ 10KHz $:

$$ \omega_o = \frac{1}{\sqrt{C_1 L}} \  \rightarrow \  2 \pi f_o = \frac{1}{\sqrt{C_1 L}} \rightarrow L \simeq 5,4 mHy $$ 

Para lograr ese valor de inductancia yo propuse:
$ Z_1 = 2K7 \Omega ; \ Z_2 = \frac{1}{s \ 1nF}; \ Z_3 = 3K \Omega \ (aunque \ 3K\Omega \ no \ es \ un \ valor \ real);\  R_4 = 150K\Omega; \ R_5 = 100K \Omega   $ 

Y por último para lograr un $Q$ de 20 bastaría con poner un Resistor de $6K8\Omega$ para estara cercano a dicho valor

Con esto puedo determinar las normas de  impedancia y frecuencia:

$Norma \ de \ pulso: \ \frac{1}{\sqrt{C_1 L}}$

$Norma \ de \ impedancia: \ \frac{1}{R}$

<img src="./item2.png" align="centre" width="1000" /> 

### Otro enfoque:

Me dí cuenta que no puedo hacerlo de esa manera, yq que no estoy tomando en cuenta la ganancia del filtro provocada por R4 y R5. Lo voy a dejar de esa manera para que se vea el error y ahora voy a buscar la transferencia con el circuito real:

Parto de los nodos del Item 1:

$Nodo \ A:$
$\frac{(V_1 - V_x)}{R} + V_x \ sC + \frac{(V_x - V_B)}{Y_1} = 0$

$Nodo \ B:$
$\frac{(V_x - V_B)}{Y_2} + \frac{(V_x - V_2)}{Y_3}  = 0$

$Nodo \ C:$
$\frac{(V_x - V_2)}{R_4} + \frac{(V_x - V_2)}{R_5}  = 0$




## Item 3

Vamos a proceder con la simulación en python del circuito normalizado

In [ ]:
# Simulación filtro pasa banda normalizado 

Z1 = 2700  
Z2 = 1 / 1e-9
Z3 = 3000
Z4 = 150000
Z5 = 100000
R  = 6800
C1 = 47e-9
L = (Z1 * Z3 * Z5) / (Z2 * Z4)

W0 = 1 / np.sqrt(L * C1)
Q = W0 * C1 * R

num = [ w0 / Q, 0]
den = [1, w0 / Q, w0**2]

H1 = signal.TransferFunction(num, den)

# Rango de frecuencias para el gráfico (1 kHz a 100 kHz)
f = np.logspace(3, 5, 1000)
w = 2 * np.pi * f
w, mag, phase = signal.bode(H1, w)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax1.semilogx(f, mag, color='blue', linewidth=2)
ax1.set_title('Diagrama de Bode - Salida $V_2$ del GIC', fontsize=14)
ax1.set_ylabel('Magnitud [dB]')
ax1.grid(True, which="both", ls="-", alpha=0.5)
ax1.axvline(f0, color='red', linestyle='--', label=f'$f_0$ = {f0/1000} kHz')
ax1.legend()

ax2.semilogx(f, phase, color='purple', linewidth=2)
ax2.set_ylabel('Fase [grados]')
ax2.set_xlabel('Frecuencia [Hz]')
ax2.grid(True, which="both", ls="-", alpha=0.5)
ax2.axvline(f0, color='red', linestyle='--')

plt.tight_layout()
pzmap(H1)
plt.show()


### Conclusión: Se verifican los requisitos del Item 2